# Benchmark cho chatbot RAG (chatbot = Groq, judge = Gemini)

**Quy trình:**
1. Cài đặt + cấu hình  →  2. Nạp tài liệu  →  3. Sinh câu hỏi test (có cả câu *không có đáp án*)  →  4. **Người rà soát file Excel**  →  5. Chạy chatbot  →  6. Chấm Retrieval  →  7. Chấm Generation (RAGAS + abstention)  →  8. Tổng hợp & phân tích lỗi

**Chuẩn bị:** thêm `GROQ_API_KEY` và `GEMINI_API_KEY` vào Colab Secrets (biểu tượng chìa khóa), bật quyền truy cập notebook. Upload tài liệu (PDF/DOCX/TXT) vào thư mục `MyDrive/rag_benchmark/docs`.

In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## 1. Cài đặt

In [ ]:
!pip install -q "ragas>=0.2" datasets pandas openpyxl docx2txt \
    langchain langchain-groq langchain-google-genai google-genai langchain-huggingface langchain-community \
    sentence-transformers faiss-cpu pypdf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.


## 2. Cấu hình
Chỉ cần sửa ô `CFG`. Mỗi lần so sánh cấu hình chatbot, **giữ nguyên judge và bộ câu hỏi**.

In [ ]:
import os, json, random, re, time
import numpy as np
import pandas as pd
from google.colab import userdata, drive

drive.mount('/content/drive')
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")   # đổi tên nếu secret của bạn đặt khác
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

BASE = "/content/drive/MyDrive/ChatboxRAG/rag_benchmark"
DOCS_DIR = f"{BASE}/docs"
os.makedirs(DOCS_DIR, exist_ok=True)

CFG = dict(
    # --- tài liệu ---
    chunk_size=800, chunk_overlap=100,
    # --- bộ câu hỏi ---
    n_answerable=60,        # số câu có đáp án trong tài liệu
    n_unanswerable=15,      # số câu KHÔNG có đáp án (chatbot phải từ chối)
    # --- judge (Gemini) ---
    judge_model="models/gemini-3.8-flash",  # alias model Flash mới nhất; có thể đổi sang tên cụ thể xem ở aistudio.google.com
    call_gap=5,                         # giây nghỉ giữa các lần gọi Gemini (tránh vượt RPM gói free)
    # --- chatbot (Groq) ---
    chatbot_model="llama-3.1-8b-instant",  # kiểm tra tên model hiện hành tại console.groq.com/docs/models
    embed_model="BAAI/bge-m3",
    top_k=5,
    run_name="baseline",    # đặt tên mỗi lần chạy: baseline, topk3, rerank...
    seed=42,
)
random.seed(CFG["seed"])
os.makedirs(f"{BASE}/runs", exist_ok=True)
print("Docs dir:", DOCS_DIR, "| files:", os.listdir(DOCS_DIR))

Mounted at /content/drive
Docs dir: /content/drive/MyDrive/ChatboxRAG/rag_benchmark/docs | files: ['L5-Phan_lop_va_Lang_gieng_gan_nhat.pdf', 'L4-Hoi_quy_va_Hoi_quy_tuyen_tinh.pdf', 'L3-Danh_gia_hieu_nang.pdf', 'L6-Hoc_dua_tren_xac_suat.pdf', 'L7-Cay_quyet_dinh.pdf', 'L8-Mang_noron_nhan_tao.pdf', 'L9-May_vecto_ho_tro.pdf', 'L10-Phan_cum_va_kMeans_va_HAC.pdf', 'L11-Phat_hien_luat_ket_hop.pdf', 'Machine Learning - Andrew Ng (PDF bản dịch tiếng Việt).pdf', 'machine-learning-algorithms_text-book.pdf', 'machine-learning-co-ban.pdf', 'tri-tue-nhan-tao_nguyen-quang-hoan_ttnt - [cuuduongthancong.com] (1).pdf', 'tri-tue-nhan-tao_tu-minh-phuong_ttnt-tu-minh-phuong-2014 - [cuuduongthancong.com].pdf', 'AI and Machine Learning for Coders A Programmers Guide to Artificial Intelligence (Laurence Moroney) 2020_infotechbooks.pdf', 'AI Engineering (for True Epub) (Chip Huyen)_infotechbooks.pdf', 'Ai_for_Student_Phien_ban_Tieng_Viet_SIHUB.pdf', 'AI_modern_approach_4th_ed_2022_Russell-Norvig_solution

## 3. Nạp và chia nhỏ tài liệu

In [ ]:
!pip install -q pymupdf

In [ ]:
import shutil, time
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

LOCAL = "/content/docs_local"
shutil.copytree(DOCS_DIR, LOCAL, dirs_exist_ok=True)   # copy từ Drive về ổ cục bộ cho nhanh

def load_docs(folder):
    docs = []
    for fn in sorted(os.listdir(folder)):
        p = os.path.join(folder, fn)
        ext = fn.lower().rsplit(".", 1)[-1]
        if ext == "pdf":    loader = PyMuPDFLoader(p)
        elif ext == "docx": loader = Docx2txtLoader(p)
        elif ext in ("txt", "md"): loader = TextLoader(p, encoding="utf-8")
        else: continue
        t0 = time.time()
        pages = loader.load()
        for d in pages:
            d.metadata["source"] = fn
        docs.extend(pages)
        chars = sum(len(d.page_content) for d in pages)
        print(f"{fn}: {len(pages)} trang, {chars:,} ký tự, {time.time()-t0:.1f}s")
    return docs

docs = load_docs(LOCAL)
splitter = RecursiveCharacterTextSplitter(chunk_size=CFG["chunk_size"], chunk_overlap=CFG["chunk_overlap"])
split = splitter.split_documents(docs)
chunks = [dict(id=f"c{i:05d}", source=d.metadata["source"], text=d.page_content) for i, d in enumerate(split)]
print(f"{len(docs)} trang/tài liệu -> {len(chunks)} chunk")

AI Engineering (for True Epub) (Chip Huyen)_infotechbooks.pdf: 1096 trang, 1,079,676 ký tự, 3.9s
AI and Machine Learning for Coders A Programmers Guide to Artificial Intelligence (Laurence Moroney) 2020_infotechbooks.pdf: 391 trang, 630,995 ký tự, 2.8s
AI-Powered Developer Build software with ChatGPT and Copilot (Nathan B. Crocker) 2024_2_infotechbooks.pdf: 242 trang, 433,349 ký tự, 1.3s
AI-Powered Developer Build software with ChatGPT and Copilot (Nathan B. Crocker) 2024_infotechbooks.pdf: 242 trang, 433,569 ký tự, 1.5s
AI_modern_approach_4th_ed_2022_Russell-Norvig_solution-manual.pdf: 585 trang, 1,208,381 ký tự, 5.2s
Ai_for_Student_Phien_ban_Tieng_Viet_SIHUB.pdf: 149 trang, 24,700 ký tự, 0.3s
Analytical Skills for AI and Data Science Building Skills for an AI-Driven Enterprise (Daniel Vaughan) 2020_infotechbooks.pdf: 244 trang, 493,566 ký tự, 0.8s
Artificial Intelligence- A Modern Approach-4th-ed.pdf: 1166 trang, 3,820,187 ký tự, 11.7s
Artificial_Intelligence_in_Finance_a_Pyt.pdf: 47

\## 4. Sinh câu hỏi test bằng Groq

- Câu **có đáp án**: sinh từ một chunk, lưu lại chunk đó làm *gold context*.
- Câu **không có đáp án**: cùng chủ đề nhưng thông tin không có trong tài liệu.
- Xoay vòng nhiều kiểu câu hỏi (tra cứu, giải thích, liệt kê, điều kiện, viết tắt/không dấu).

⚠️ Câu hỏi sinh bằng LLM **phải được người rà soát** ở bước kế tiếp.

In [ ]:
for m in groq_client.models.list().data:
    print(m.id)

openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3
canopylabs/orpheus-v1-english
allam-2-7b
qwen/qwen3.8-27b
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-arabic-saudi
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-20b


In [ ]:
from groq import Groq
from tqdm.auto import tqdm

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"], timeout=30.0, max_retries=0)

# ---- chọn model Groq (tự động từ danh sách key của bạn dùng được) ----
GROQ_GEN_MODEL = "openai/gpt-oss-120b"   # muốn tự chọn thì điền tên model vào đây, để None là tự chọn
SKIP = ("whisper", "tts", "guard", "orpheus", "embed", "playai", "safeguard", "compound")
PREFER = ["gpt-oss-120b", "llama-3.3-70b", "qwen3-32b", "qwen", "llama-4", "gpt-oss-20b", "llama-3.1-8b"]
avail = [m.id for m in groq_client.models.list().data if not any(k in m.id.lower() for k in SKIP)]
print("Model chat khả dụng:", avail)
if GROQ_GEN_MODEL is None:
    GROQ_GEN_MODEL = next((m for k in PREFER for m in avail if k in m.lower()), avail[0])
print("=> Dùng model:", GROQ_GEN_MODEL)

GROQ_GAP = 2.5          # giây nghỉ giữa các lần gọi (tránh vượt giới hạn RPM)
_use_json_mode = True
_fail_streak = 0

QTYPES = [
    "tra cứu sự kiện/con số cụ thể",
    "giải thích khái niệm hoặc quy trình",
    "liệt kê các mục/bước/điều kiện",
    "tình huống áp dụng điều kiện (nếu... thì...)",
    "câu hỏi ngắn kiểu người dùng thật: viết tắt, thiếu dấu hoặc thiếu chủ ngữ",
]

def _parse(txt):
    txt = re.sub(r"<think>.*?</think>", "", txt, flags=re.S).strip()
    i, j = txt.find("{"), txt.rfind("}")
    return json.loads(txt[i:j + 1])

def chat_json(prompt, temp=0.7, retries=5):
    global _use_json_mode, _fail_streak
    for a in range(retries):
        try:
            kw = dict(model=GROQ_GEN_MODEL, temperature=temp,
                      messages=[{"role": "user", "content": prompt}])
            if _use_json_mode:
                kw["response_format"] = {"type": "json_object"}
            r = groq_client.chat.completions.create(**kw)
            out = _parse(r.choices[0].message.content)
            time.sleep(GROQ_GAP)
            _fail_streak = 0
            return out
        except Exception as e:
            msg = f"{type(e).__name__}: {e}"
            low = msg.lower()
            if any(k in low for k in ("404", "does not exist", "invalid_api_key", "401")):
                raise RuntimeError("Lỗi cấu hình Groq (sai tên model hoặc key), dừng lại: " + msg[:250])
            if "400" in msg and ("response_format" in low or "json" in low) and _use_json_mode:
                _use_json_mode = False
                print("[Groq] model không hỗ trợ JSON mode, chuyển sang parse thủ công")
                continue
            if isinstance(e, ValueError):      # JSON hỏng -> thử lại nhanh
                print(f"[Groq] JSON lỗi, thử lại {a+1}/{retries}")
                time.sleep(1); continue
            wait = 15 * (a + 1) if "429" in msg else 5 * (a + 1)
            print(f"[Groq] lỗi {a+1}/{retries}: {msg[:90]} - chờ {wait}s")
            time.sleep(wait)
    _fail_streak += 1
    if _fail_streak >= 8:
        raise RuntimeError("Groq lỗi liên tục 8 câu liền, dừng lại để bạn kiểm tra.")
    return None

# ---- kiểm tra nhanh trước khi chạy cả loạt ----
t0 = time.time()
print("Test:", chat_json('Trả về JSON: {"ok": true}', temp=0), f"({time.time()-t0:.1f}s)")

def gen_answerable(chunk, qtype):
    p = ("Bạn tạo bộ câu hỏi kiểm thử cho chatbot hỏi đáp tài liệu.\n"
         "Dựa DUY NHẤT vào đoạn văn dưới đây, viết 1 câu hỏi tiếng Việt thuộc loại: " + qtype + ".\n"
         "Yêu cầu: câu hỏi phải hiểu được khi đứng một mình (không nói 'đoạn văn trên'), "
         "trả lời được chỉ bằng đoạn này, và kèm đáp án chuẩn đầy đủ, ngắn gọn.\n"
         'Trả về JSON: {"question": "...", "ground_truth": "..."}\n\n'
         "ĐOẠN VĂN:\n" + chunk["text"])
    return chat_json(p)

def gen_unanswerable(chunk):
    p = ("Bạn tạo bộ câu hỏi kiểm thử cho chatbot hỏi đáp tài liệu.\n"
         "Dựa vào chủ đề của đoạn văn dưới đây, viết 1 câu hỏi tiếng Việt CÙNG CHỦ ĐỀ nhưng hỏi về "
         "thông tin KHÔNG có trong đoạn này và nhiều khả năng cũng không có trong tài liệu "
         "(ví dụ một con số, ngày, điều khoản, chi tiết cụ thể không được nhắc tới). "
         "Câu hỏi phải nghe hợp lý, không được vô lý.\n"
         'Trả về JSON: {"question": "..."}\n\n'
         "ĐOẠN VĂN:\n" + chunk["text"])
    return chat_json(p)

# ---- chọn chunk (cố định theo seed) ----
usable = [c for c in chunks if len(c["text"]) >= 250]
random.Random(CFG["seed"]).shuffle(usable)
need = CFG["n_answerable"] + CFG["n_unanswerable"]
assert len(usable) >= need, f"Chỉ có {len(usable)} chunk đủ dài, cần {need}. Giảm n_answerable/n_unanswerable hoặc thêm tài liệu."
pool = usable[:need]
pool_ans, pool_una = pool[:CFG["n_answerable"]], pool[CFG["n_answerable"]:]
wanted = {(True, c["id"]) for c in pool_ans} | {(False, c["id"]) for c in pool_una}

# ---- cache: lưu từng câu ra Drive, chạy lại sẽ tiếp tục ----
cache_path = f"{BASE}/gen_cache.jsonl"
rows = [json.loads(l) for l in open(cache_path, encoding="utf-8")] if os.path.exists(cache_path) else []
done = {(r["answerable"], r["seed_chunk_id"]) for r in rows}
print(f"Cache: đã có {len(rows)} câu")

def save_row(r):
    rows.append(r); done.add((r["answerable"], r["seed_chunk_id"]))
    with open(cache_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

for i, ch in enumerate(tqdm(pool_ans, desc="Câu có đáp án")):
    if (True, ch["id"]) in done: continue
    qt = QTYPES[i % len(QTYPES)]
    out = gen_answerable(ch, qt)
    if out and out.get("question") and out.get("ground_truth"):
        save_row(dict(question=out["question"], ground_truth=out["ground_truth"], answerable=True,
                      gold_chunk_id=ch["id"], gold_text=ch["text"], source=ch["source"], qtype=qt,
                      seed_chunk_id=ch["id"], gen_by=GROQ_GEN_MODEL))

for ch in tqdm(pool_una, desc="Câu không có đáp án"):
    if (False, ch["id"]) in done: continue
    out = gen_unanswerable(ch)
    if out and out.get("question"):
        save_row(dict(question=out["question"], ground_truth="", answerable=False,
                      gold_chunk_id="", gold_text="", source=ch["source"], qtype="không có đáp án",
                      seed_chunk_id=ch["id"], gen_by=GROQ_GEN_MODEL))

rows = [r for r in rows if (r["answerable"], r["seed_chunk_id"]) in wanted]
bench = pd.DataFrame(rows)
bench.insert(0, "id", [f"q{i:03d}" for i in range(len(bench))])
bench["keep"] = 1   # người rà soát đổi thành 0 nếu câu hỏi tệ
bench["note"] = ""
draft_path = f"{BASE}/benchmark_draft.xlsx"
bench.to_excel(draft_path, index=False)
print(bench.answerable.value_counts().to_dict(), "| sinh bởi:", bench.gen_by.value_counts().to_dict(), "->", draft_path)
bench.head()

Model chat khả dụng: ['qwen/qwen3.8-27b', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'allam-2-7b']
=> Dùng model: openai/gpt-oss-120b
Test: {'ok': True} (3.0s)
Cache: đã có 75 câu


Câu có đáp án:   0%|          | 0/60 [00:00<?, ?it/s]

Câu không có đáp án:   0%|          | 0/15 [00:00<?, ?it/s]

{True: 60, False: 15} | sinh bởi: {'openai/gpt-oss-120b': 75} -> /content/drive/MyDrive/ChatboxRAG/rag_benchmark/benchmark_draft.xlsx


,id,question,ground_truth,answerable,gold_chunk_id,gold_text,source,qtype,seed_chunk_id,gen_by,keep,note
0,q000,Công thức kết tập nhãn trong đoạn văn tính tru...,k,True,c20553,Học máy và khai phá dữliệu - Tài liệu đọc\ntín...,IT3190-Tai-lieu-doc.pdf,tra cứu sự kiện/con số cụ thể,c20553,openai/gpt-oss-120b,1,
1,q001,Quá trình loại trừ biến T trong thuật toán suy...,"Ta tìm các thừa số trong tập F chứa biến T, cụ...",True,c25901,"Suy diễn xác suất \n \n142\n \nTrước hết, xác ...",tri-tue-nhan-tao_tu-minh-phuong_ttnt-tu-minh-p...,giải thích khái niệm hoặc quy trình,c25901,openai/gpt-oss-120b,1,
2,q002,"Trong đoạn văn, liệt kê các khía cạnh liên qua...",Sử dụng file cấu hình để triển khai nhiều mô h...,True,c02896,"a configuration file, examining how to use it ...",AI and Machine Learning for Coders A Programme...,liệt kê các mục/bước/điều kiện,c02896,openai/gpt-oss-120b,1,
3,q003,Nếu muốn tìm một chủ đề mô tả các bộ phim lịch...,Topic 1,True,c21858,pretty kid performance joe ...,Introduction to Machine Learning with Python (...,tình huống áp dụng điều kiện (nếu... thì...),c21858,openai/gpt-oss-120b,1,
4,q004,Ai biết bò?,Steve,True,c25778,Biểu diễn tri thức và suy diễn logic \n \n106\...,tri-tue-nhan-tao_tu-minh-phuong_ttnt-tu-minh-p...,"câu hỏi ngắn kiểu người dùng thật: viết tắt, t...",c25778,openai/gpt-oss-120b,1,


In [ ]:
# ===== SINH THÊM 1 ĐỢT CÂU HỎI (tái sử dụng được nhiều lần) =====
import glob
BATCH_NAME = "bm_extra2"              # đợt trước là "extra", đợt sau đổi thành "extra3", "extra4"...
GROQ_GEN_MODEL = "qwen/qwen3.8-27b"       # điền tên model Groq muốn dùng
N_EXTRA_ANS, N_EXTRA_UNA = 60, 15  # 75 câu

x_cache = f"{BASE}/gen_cache_{BATCH_NAME}.jsonl"

# 1) loại chunk đã dùng ở mọi đợt trước (trừ cache của chính đợt này, để chạy lại thì tiếp tục đúng chỗ)
used = set()
for p in glob.glob(f"{BASE}/gen_cache*.jsonl"):
    if os.path.abspath(p) == os.path.abspath(x_cache): continue
    for l in open(p, encoding="utf-8"):
        if l.strip(): used.add(json.loads(l)["seed_chunk_id"])
free = [c for c in chunks if len(c["text"]) >= 250 and c["id"] not in used]
random.Random(CFG["seed"] + sum(map(ord, BATCH_NAME))).shuffle(free)
print(f"Đã dùng {len(used)} chunk ở các đợt trước | còn {len(free)} chunk trống")

if len(free) < N_EXTRA_ANS + N_EXTRA_UNA:
    ratio = N_EXTRA_UNA / (N_EXTRA_ANS + N_EXTRA_UNA)
    N_EXTRA_UNA = int(len(free) * ratio)
    N_EXTRA_ANS = len(free) - N_EXTRA_UNA
    print(f"⚠️ Không đủ chunk, giảm xuống {N_EXTRA_ANS} có đáp án + {N_EXTRA_UNA} không có đáp án")
x_ans = free[:N_EXTRA_ANS]
x_una = free[N_EXTRA_ANS:N_EXTRA_ANS + N_EXTRA_UNA]

# 2) cache riêng cho đợt này
x_rows = [json.loads(l) for l in open(x_cache, encoding="utf-8")] if os.path.exists(x_cache) else []
x_done = {(r["answerable"], r["seed_chunk_id"]) for r in x_rows}
print(f"Đợt '{BATCH_NAME}': cache đã có {len(x_rows)} câu | model: {GROQ_GEN_MODEL}")

def x_save(r):
    x_rows.append(r); x_done.add((r["answerable"], r["seed_chunk_id"]))
    with open(x_cache, "a", encoding="utf-8") as f:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

for i, ch in enumerate(tqdm(x_ans, desc="Có đáp án")):
    if (True, ch["id"]) in x_done: continue
    qt = QTYPES[i % len(QTYPES)]
    out = gen_answerable(ch, qt)
    if out and out.get("question") and out.get("ground_truth"):
        x_save(dict(question=out["question"], ground_truth=out["ground_truth"], answerable=True,
                    gold_chunk_id=ch["id"], gold_text=ch["text"], source=ch["source"], qtype=qt,
                    seed_chunk_id=ch["id"], gen_by=GROQ_GEN_MODEL))

for ch in tqdm(x_una, desc="Không có đáp án"):
    if (False, ch["id"]) in x_done: continue
    out = gen_unanswerable(ch)
    if out and out.get("question"):
        x_save(dict(question=out["question"], ground_truth="", answerable=False,
                    gold_chunk_id="", gold_text="", source=ch["source"], qtype="không có đáp án",
                    seed_chunk_id=ch["id"], gen_by=GROQ_GEN_MODEL))

# 3) xuất file riêng; id mang tên đợt để không trùng id các đợt khác
want_x = {(True, c["id"]) for c in x_ans} | {(False, c["id"]) for c in x_una}
extra = pd.DataFrame([r for r in x_rows if (r["answerable"], r["seed_chunk_id"]) in want_x])
extra.insert(0, "id", [f"{BATCH_NAME}_{i:03d}" for i in range(len(extra))])
extra["keep"] = 1
extra["note"] = ""
out_path = f"{BASE}/benchmark_{BATCH_NAME}.xlsx"
extra.to_excel(out_path, index=False)
print(len(extra), "câu |", extra.answerable.value_counts().to_dict(), "| sinh bởi:", extra.gen_by.value_counts().to_dict(), "->", out_path)

Đã dùng 150 chunk ở các đợt trước | còn 23996 chunk trống
Đợt 'bm_extra2': cache đã có 75 câu | model: qwen/qwen3.8-27b


Có đáp án:   0%|          | 0/60 [00:00<?, ?it/s]

Không có đáp án:   0%|          | 0/15 [00:00<?, ?it/s]

75 câu | {True: 60, False: 15} | sinh bởi: {'qwen/qwen3.8-27b': 75} -> /content/drive/MyDrive/ChatboxRAG/rag_benchmark/benchmark_bm_extra2.xlsx


## 5. ✋ Rà soát bằng người (bắt buộc)

Mở `benchmark_draft.xlsx` trên Drive và:
- Đặt `keep = 0` cho câu hỏi mơ hồ, sai, quá dễ, hoặc không tự nhiên.
- Sửa `question` / `ground_truth` nếu cần.
- Với câu `answerable = False`: **kiểm tra thật sự không có đáp án trong tài liệu**. Nếu có thì xóa hoặc đặt `keep = 0`.
- Bổ sung thêm câu hỏi thật từ log người dùng nếu có (thêm dòng mới, điền đủ cột).

Lưu lại thành **`benchmark_reviewed.xlsx`** trong cùng thư mục, rồi chạy tiếp.

In [ ]:
from google.colab import files

reviewed_path = f"{BASE}/benchmark_reviewed.xlsx"

# ===== TÙY CHỌN =====
RUN_ALL      = True    # True: chạy TOÀN BỘ câu hỏi trong file. False: chạy thử một phần (theo 2 số bên dưới)
UPLOAD_NEW   = False   # True: luôn hiện nút chọn file để thả file mới. False: dùng file đã có trên Drive (chưa có thì tự hỏi)
DROP_SUSPECT = False   # True: tự loại câu nghi ngờ (nhắc "đoạn văn" hoặc đáp án <= 3 ký tự)
N_TRUE_TEST  = False      # chỉ dùng khi RUN_ALL = False: số câu CÓ đáp án
N_FALSE_TEST = False      # chỉ dùng khi RUN_ALL = False: số câu KHÔNG có đáp án

# 1) thả file vào (chỉ khi cần), lưu một bản lên Drive để các lần sau dùng lại
if UPLOAD_NEW or not os.path.exists(reviewed_path):
    up = files.upload()
    assert up, "Bạn chưa chọn file nào."
    name = list(up.keys())[0]
    with open(reviewed_path, "wb") as f:
        f.write(up[name])
    print("Đã nhận:", name, "-> lưu tại", reviewed_path)
else:
    print("Dùng file có sẵn trên Drive:", reviewed_path)

# 2) đọc và kiểm tra
bench = pd.read_excel(reviewed_path)
missing = [c for c in ["id", "question", "ground_truth", "answerable", "keep"] if c not in bench.columns]
assert not missing, f"File thiếu cột bắt buộc: {missing}"
for c in ["gold_text", "gold_chunk_id", "qtype", "source", "note"]:
    if c not in bench.columns:
        bench[c] = ""
bench = bench.fillna({"ground_truth": "", "gold_text": "", "gold_chunk_id": "", "qtype": "khác", "source": "", "note": ""})

# answerable về True/False (chữ "FALSE" dạng text sẽ bị astype(bool) hiểu nhầm thành True)
bench["answerable"] = bench["answerable"].astype(str).str.strip().str.upper().isin(["TRUE", "1", "1.0", "YES"])
bench["keep"] = pd.to_numeric(bench["keep"], errors="coerce").fillna(1)
bench = bench[(bench["keep"] != 0) & (bench["question"].astype(str).str.strip() != "")].reset_index(drop=True)
bench["id"] = bench["id"].astype(str)
if bench["id"].duplicated().any():
    print("⚠️ id bị trùng, tự đánh lại id")
    bench["id"] = [f"q{i:04d}" for i in range(len(bench))]

# 3) cảnh báo các lỗi hay gặp
no_gt = (bench.answerable) & (bench.ground_truth.astype(str).str.strip() == "")
no_gold = (bench.answerable) & (bench.gold_text.astype(str).str.strip() == "")
has_gt = (~bench.answerable) & (bench.ground_truth.astype(str).str.strip() != "")
print("Câu có đáp án nhưng ground_truth trống:", int(no_gt.sum()))
print("Câu có đáp án nhưng thiếu gold_text (bỏ qua điểm retrieval):", int(no_gold.sum()))
print("Câu KHÔNG có đáp án nhưng vẫn có ground_truth:", int(has_gt.sum()))

pat = r"(?:đoạn văn|đoạn trên|trong đoạn|theo đoạn|văn bản trên)"
suspect = bench.question.str.lower().str.contains(pat) | (bench.answerable & (bench.ground_truth.astype(str).str.strip().str.len() <= 3))
print(f"Câu nghi ngờ (nhắc 'đoạn văn' hoặc đáp án quá ngắn): {int(suspect.sum())}")
if DROP_SUSPECT:
    bench = bench[~suspect].reset_index(drop=True)
    print("-> đã loại các câu nghi ngờ")

# 4) lấy mẫu chạy thử (chỉ khi RUN_ALL = False; cố định theo seed để lần sau ra cùng bộ)
if not RUN_ALL and (N_TRUE_TEST or N_FALSE_TEST):
    t = bench[bench.answerable]; f = bench[~bench.answerable]
    if N_TRUE_TEST:  t = t.sample(min(N_TRUE_TEST, len(t)), random_state=CFG["seed"])
    if N_FALSE_TEST: f = f.sample(min(N_FALSE_TEST, len(f)), random_state=CFG["seed"])
    bench = pd.concat([t, f]).sample(frac=1, random_state=CFG["seed"]).reset_index(drop=True)
    print("⚠️ ĐANG CHẠY THỬ MỘT PHẦN. Đặt RUN_ALL = True để chạy toàn bộ.")
else:
    print("✅ Chạy TOÀN BỘ câu hỏi trong file.")

print(len(bench), "câu hỏi sẽ chạy |", bench.answerable.value_counts().to_dict())

Dùng file có sẵn trên Drive: /content/drive/MyDrive/ChatboxRAG/rag_benchmark/benchmark_reviewed.xlsx
Câu có đáp án nhưng ground_truth trống: 0
Câu có đáp án nhưng thiếu gold_text (bỏ qua điểm retrieval): 0
Câu KHÔNG có đáp án nhưng vẫn có ground_truth: 0
Câu nghi ngờ (nhắc 'đoạn văn' hoặc đáp án quá ngắn): 6
✅ Chạy TOÀN BỘ câu hỏi trong file.
56 câu hỏi sẽ chạy | {True: 41, False: 15}


## 6. Chatbot cần test

Mục này đã được nối với **chatbot thật (ChatboxRAG)**: nạp FAISS đã build sẵn trên Drive, hybrid BM25 + vector → reranker `bge-reranker-v2-m3` → Groq `gpt-oss-120b`, prompt chép nguyên văn từ `rag_engine_web.py`.

- `ask_rag(question)` trả về `{"answer": str, "contexts": [str, ...]}` — đúng định dạng mục 7 cần.
- Nếu đổi cấu hình trên web (model, `RETRIEVE_N`, `RERANK_TOP_K`, prompt) thì sửa tương ứng ở cell dưới để benchmark khớp với web.
- Cell cuối của mục này chỉ để tạo `emb` cho RAGAS (judge), không phải embedding của chatbot.


In [ ]:
!pip install -q langchain-classic rank_bm25 langchain-huggingface sentence-transformers faiss-cpu langchain-groq


In [ ]:
# ============================================================
# KẾT NỐI CHATBOT THẬT (ChatboxRAG): hybrid BM25 + FAISS -> reranker -> Groq
# Điều kiện: Drive đã mount và index đã build sẵn ở notebook chatbot.
# ============================================================
import os, re, time, numpy as np, torch, faiss
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# ---------- tham số ----------
INDEX_PATH   = "/content/drive/MyDrive/ChatboxRAG/faiss_index_huggingface"   # chỉ mục cần test
GROQ_MODEL   = "openai/gpt-oss-120b"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
RETRIEVE_N   = 20      # ứng viên mỗi nhánh (BM25 / vector)
RERANK_TOP_K = 5       # số chunk đưa vào LLM
CLEAN_CONTEXTS = True  # True: contexts trả cho benchmark đã bỏ header "Tài liệu: ..." và <!-- page: N -->

EMBED_MODEL = None     # None: tự chọn theo số chiều của chỉ mục (khuyên dùng). Hoặc điền tên model để ép dùng
RUN_NAME    = None     # None: tự đặt theo model embedding. Hoặc điền tên riêng (vd "bge_m3_v2")

# các model embedding mà chatbot có thể đã dùng để build chỉ mục
EMBED_CFG = {
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": dict(dim=384,  encode={}, max_len=None, tag="minilm"),
    "BAAI/bge-m3":                                                  dict(dim=1024, encode={"normalize_embeddings": True}, max_len=1024, tag="bge_m3"),
}

# ---------- đọc số chiều của chỉ mục và chọn model embedding khớp ----------
index_file = os.path.join(INDEX_PATH, "index.faiss")
assert os.path.exists(index_file), f"Không thấy index tại {INDEX_PATH}"
idx_dim = faiss.read_index(index_file).d
print(f"Chỉ mục tại {INDEX_PATH}: {idx_dim} chiều")

if EMBED_MODEL is None:
    cands = [n for n, c in EMBED_CFG.items() if c["dim"] == idx_dim]
    assert cands, (f"Chỉ mục {idx_dim} chiều không khớp model nào trong EMBED_CFG. "
                   f"Thêm model đã dùng để build chỉ mục vào EMBED_CFG, hoặc dùng chỉ mục khác.")
    EMBED_MODEL = cands[0]
    print("=> Tự chọn model embedding:", EMBED_MODEL)
else:
    assert EMBED_MODEL in EMBED_CFG, f"{EMBED_MODEL} chưa có trong EMBED_CFG"
    assert EMBED_CFG[EMBED_MODEL]["dim"] == idx_dim, (
        f"LỆCH: {EMBED_MODEL} tạo vector {EMBED_CFG[EMBED_MODEL]['dim']} chiều, chỉ mục lại {idx_dim} chiều. "
        f"Đổi EMBED_MODEL hoặc đổi INDEX_PATH cho khớp.")
ec = EMBED_CFG[EMBED_MODEL]

# ghi cấu hình thật vào CFG để history.csv đúng với chatbot được test
CFG["chatbot_model"] = GROQ_MODEL
CFG["embed_model"]   = EMBED_MODEL
CFG["top_k"]         = RERANK_TOP_K
CFG["chunk_size"]    = 1500           # chatbot thật cắt chunk 1500/300 (trước đó bảng ghi nhầm 800)
CFG["run_name"]      = RUN_NAME or f"{ec['tag']}_hybrid_rerank"
print("run_name:", CFG["run_name"])

# ---------- nạp embedding + index + dựng hybrid retriever ----------
dev = "cuda" if torch.cuda.is_available() else "cpu"
bot_emb = HuggingFaceEmbeddings(model_name=EMBED_MODEL, model_kwargs={"device": dev}, encode_kwargs=ec["encode"])
if ec["max_len"]:
    bot_emb._client.max_seq_length = ec["max_len"]
q_dim = len(bot_emb.embed_query("thử"))
assert q_dim == idx_dim, f"Vector câu hỏi {q_dim} chiều nhưng chỉ mục {idx_dim} chiều"

vs = FAISS.load_local(INDEX_PATH, bot_emb, allow_dangerous_deserialization=True)
bot_chunks = list(vs.docstore._dict.values())
print("Đã nạp FAISS:", vs.index.ntotal, "vector |", len(bot_chunks), "chunk")

bm25 = BM25Retriever.from_documents(bot_chunks, preprocess_func=lambda t: t.lower().split())
bm25.k = RETRIEVE_N
vector_ret = vs.as_retriever(search_type="similarity", search_kwargs={"k": RETRIEVE_N})
hybrid = EnsembleRetriever(retrievers=[bm25, vector_ret], weights=[0.3, 0.7])

RERANKER = CrossEncoder(RERANK_MODEL, max_length=1024, device=dev)
print("Đã nạp reranker:", RERANK_MODEL)

def retrieve_with_rerank(question, k=RERANK_TOP_K):
    cands = hybrid.invoke(question)
    uniq = {}
    for d in cands:                       # khử trùng lặp giữa 2 nhánh
        m = d.metadata
        key = (m.get("doc_id"), m.get("start_index"), m.get("page_start"), m.get("page_end"))
        uniq.setdefault(key, d)
    cands = list(uniq.values())
    if not cands:
        return []
    scores = RERANKER.predict([(question, d.page_content) for d in cands], batch_size=16, show_progress_bar=False)
    order = np.argsort(-np.asarray(scores))[:k]
    return [cands[i] for i in order]

# ---------- LLM + prompt (chép nguyên văn từ rag_engine_web.py) ----------
TEMPLATE = """
Bạn là trợ lý học tập môn Trí tuệ nhân tạo.

NHIỆM VỤ:
Giải đáp câu hỏi của sinh viên dựa trên tài liệu được cung cấp.

QUY TẮC:
1. Chỉ sử dụng thông tin trong NGỮ CẢNH.
2. Không tự ý bổ sung kiến thức bên ngoài, không đoán.
3. Nếu NGỮ CẢNH không có thông tin để trả lời, chỉ nói ngắn gọn (bằng ngôn ngữ của câu hỏi)
   rằng tài liệu môn học không đề cập nội dung này.
4. Nếu NGỮ CẢNH chỉ có một phần thông tin, hãy trả lời phần có và nói rõ phần nào NGỮ CẢNH chưa đủ để trả lời.
   KHÔNG kết luận tài liệu môn học thiếu nội dung đó; với câu hỏi tóm tắt nhiều chương, gợi ý hỏi từng chương để xem phần còn lại.
5. Giải thích dễ hiểu, phù hợp với sinh viên.
6. Có thể sử dụng ví dụ nếu ví dụ đó có trong tài liệu.
7. Trả lời bằng ngôn ngữ của câu hỏi.
8. Sau mỗi nội dung có sử dụng thông tin tài liệu, ghi nguồn theo định dạng (Tên tài liệu, trang X-Y),
   chép đúng theo nhãn trong ngoặc vuông ở đầu mỗi đoạn NGỮ CẢNH.
   Dòng "<!-- page: N -->" (nếu có) cho biết nội dung ngay sau nó thuộc trang N.
9. Công thức toán phải viết bằng plain text dễ đọc, KHÔNG dùng LaTeX hay cặp $...$ / $$...$$.
   Ví dụ: sigmoid(x) = 1 / (1 + exp(-x)); softmax(x_i) = exp(x_i) / tong_cac_j(exp(x_j)); P(A|B) = P(B|A) * P(A) / P(B).
   Dùng ký tự thường, dấu ngoặc và phép toán rõ ràng để sinh viên đọc hiểu ngay.
10. Phạm vi môn là Trí tuệ nhân tạo. Nếu câu hỏi hỏi kiến thức chung ngoài môn (địa lý, thủ đô, lịch sử...),
    KHÔNG trả lời dù NGỮ CẢNH tình cờ chứa câu chữ giống đáp án (vd: "Paris là thủ đô nước Pháp" chỉ là ví dụ
    logic mệnh đề, không phải nội dung môn). Khi đó từ chối ngắn gọn rằng tài liệu môn học
    không đề cập nội dung này (câu hỏi ngoài phạm vi môn).

NGỮ CẢNH:
{context}

CÂU HỎI:
{question}

TRẢ LỜI:
"""

def strip_think(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()

llm = ChatGroq(model=GROQ_MODEL, temperature=0)
prompt = ChatPromptTemplate.from_template(TEMPLATE)
chain = prompt | llm | StrOutputParser() | RunnableLambda(strip_think)

def page_label(meta):
    s, e = meta.get("page_start"), meta.get("page_end")
    if s is None: return "trang ?"
    return f"trang {s}" if s == e else f"trang {s}-{e}"

def _src_label(d):
    m = d.metadata
    return f"[{m.get('doc_title') or m.get('doc_id', 'Unknown')} ({m.get('doc_id', 'Unknown')}), {page_label(m)}]"

def format_docs(docs):                    # đúng như web: LLM nhìn thấy ĐẦY ĐỦ nhãn + header
    return "\n\n".join(f"{_src_label(d)}\n{d.page_content}" for d in docs)

_HDR  = re.compile(r"^(?:Chương \d+ \(elc3008_ch\d+\)\n)?Tài liệu: [^\n]*\n")
_PAGE = re.compile(r"\n?<!-- page: \d+ -->\n?")
def clean_ctx(t):
    return _PAGE.sub("\n", _HDR.sub("", t, count=1)).strip()

# ---------- hàm benchmark gọi: trả về {"answer", "contexts"} ----------
def ask_rag(question):
    docs = retrieve_with_rerank(question)
    context = format_docs(docs) or "(Không tìm thấy đoạn tài liệu nào phù hợp.)"
    for attempt in range(3):
        try:
            answer = chain.invoke({"context": context, "question": question})
            break
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower():
                wait = 15 * (attempt + 1)
                print(f"Groq báo giới hạn tốc độ, chờ {wait}s rồi thử lại...")
                time.sleep(wait)
            else:
                raise
    else:
        raise RuntimeError("Groq vẫn báo giới hạn sau 3 lần thử, hãy chờ vài phút rồi chạy lại.")
    ctxs = [f"{_src_label(d)}\n{clean_ctx(d.page_content)}" if CLEAN_CONTEXTS else d.page_content for d in docs]
    return {"answer": answer, "contexts": ctxs}

ask_chatbot = ask_rag   # tên gốc mà mục 7 dùng

print(f"Sẵn sàng: LLM = {GROQ_MODEL} | embedding = {EMBED_MODEL} | hybrid {RETRIEVE_N}/nhánh → rerank top-{RERANK_TOP_K}")
test = ask_rag(bench.question.iloc[0])
print("\nCÂU HỎI:", bench.question.iloc[0])
print("TRẢ LỜI:", test["answer"][:500])
print("CONTEXT #1:", test["contexts"][0][:300])

Chỉ mục tại /content/drive/MyDrive/ChatboxRAG/faiss_index_huggingface: 1024 chiều
=> Tự chọn model embedding: BAAI/bge-m3
run_name: bge_m3_hybrid_rerank


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

... (18 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Đã nạp reranker: BAAI/bge-reranker-v2-m3
Sẵn sàng: LLM = openai/gpt-oss-120b | embedding = BAAI/bge-m3 | hybrid 20/nhánh → rerank top-5

CÂU HỎI: Trong thuật toán K-nearest neighbors (KNN) dùng cho hồi quy, làm thế nào để dự đoán giá trị đầu ra từ K hàng xóm gần nhất?
TRẢ LỜI: Trong K‑nearest neighbors (KNN) áp dụng cho bài toán hồi quy, giá trị dự đoán của một điểm mới **x** được tính dựa trên các giá trị đầu ra của K điểm lân cận nhất (x₁,…,x_K) và có thể thực hiện theo 2 cách chính:

1. **Trung bình cộng (không trọng số)**  
   Khi không dùng trọng số, ta lấy trung bình của các nhãn y₁,…,y_K của K hàng xóm:  

   ŷ = (1 / K) * Σ_{i=1..K} y_i  

   (Nguồn: IT3190 — Tài liệu đọc, trang 60)

2. **Trung bình có trọng số**  
   Nếu muốn các hàng xóm gần hơn có ảnh hưởn
CONTEXT #1: [Machine Learning cơ bản (ml_co_ban), trang 126]
Hình 9.3. KNN cho bài toán hồi quy (Nguồn: Nearest neighbors regression – scikitlearn – https://goo.gl/9VyBF3). Kết quả:

Accuracy of 7NN (customized weights): 

In [ ]:
# Embedding CHỈ để RAGAS dùng cho ResponseRelevancy (không liên quan chatbot).
# Giữ bge-m3 như cũ để điểm judge so sánh được với các lần chạy trước.
from langchain_huggingface import HuggingFaceEmbeddings
emb = HuggingFaceEmbeddings(model_name="BAAI/bge-m3",
                            encode_kwargs={"normalize_embeddings": True, "batch_size": 64})


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## 7. Chạy chatbot trên toàn bộ bộ câu hỏi
Có checkpoint: nếu Colab ngắt hoặc dính rate limit, chạy lại ô này sẽ tiếp tục từ chỗ dở.

In [ ]:
import re, time, json
from tqdm.auto import tqdm

MIN_GAP = 25        # giây nghỉ giữa các câu; gặp lỗi 429 nhiều thì tăng, ít thì giảm
MAX_ATTEMPTS = 6

run_file = f"{BASE}/runs/{CFG['run_name']}_outputs.jsonl"
done = {}
if os.path.exists(run_file):
    for line in open(run_file, encoding="utf-8"):
        if line.strip():
            r = json.loads(line); done[r["id"]] = r
print(f"Run '{CFG['run_name']}': đã có {len(done)}/{len(bench)} câu")

def ask_once(question):        # như ask_rag nhưng KHÔNG tự thử lại, để vòng lặp bên dưới quản lý
    docs = retrieve_with_rerank(question)
    context = format_docs(docs) or "(Không tìm thấy đoạn tài liệu nào phù hợp.)"
    answer = chain.invoke({"context": context, "question": question})
    ctxs = [f"{_src_label(d)}\n{clean_ctx(d.page_content)}" if CLEAN_CONTEXTS else d.page_content for d in docs]
    return {"answer": answer, "contexts": ctxs}

def wait_from_error(msg, default):
    m = re.search(r"try again in (?:(\d+)h)?\s*(?:(\d+)m)?\s*([\d.]+)s", msg)
    if m:
        return int(m.group(1) or 0) * 3600 + int(m.group(2) or 0) * 60 + float(m.group(3)) + 2
    return default

stop_all = False
with open(run_file, "a", encoding="utf-8") as f:
    pbar = tqdm(list(bench.iterrows()), desc="Hỏi chatbot")
    for _, row in pbar:
        if stop_all: break
        if row["id"] in done: continue
        for attempt in range(MAX_ATTEMPTS):
            try:
                t0 = time.time()
                out = ask_once(row["question"])
                lat = time.time() - t0            # chỉ tính lần gọi thành công, không tính thời gian chờ
                rec = dict(id=row["id"], answer=out["answer"], contexts=out["contexts"], latency=lat)
                f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
                done[row["id"]] = rec
                pbar.set_postfix(giay=round(lat, 1), xong=len(done))
                break
            except Exception as e:
                msg = str(e); low = msg.lower()
                if "404" in msg or "model_not_found" in msg or "401" in msg:
                    raise RuntimeError("Sai tên model hoặc key Groq, dừng lại: " + msg[:200])
                if "429" in msg or "rate_limit" in low:
                    wait = wait_from_error(msg, 30 * (attempt + 1))
                    if "per day" in low or "(tpd)" in low or wait > 600:
                        print(f"\n⛔ Có vẻ hết hạn mức theo NGÀY (cần chờ khoảng {wait/60:.0f} phút). "
                              f"Dừng lại; chạy lại ô này sau, các câu đã xong được giữ nguyên.")
                        stop_all = True
                        break
                    print(f"{row['id']}: Groq giới hạn, chờ {wait:.0f}s (lần {attempt + 1}/{MAX_ATTEMPTS})")
                    time.sleep(wait)
                else:
                    print(row["id"], "lỗi:", msg[:120], "- thử lại"); time.sleep(10 * (attempt + 1))
        time.sleep(MIN_GAP)

res = pd.DataFrame(done.values())
df = bench.merge(res, on="id", how="left")
print(f"Đã có kết quả cho {df.answer.notna().sum()}/{len(df)} câu")

Run 'bge_m3_hybrid_rerank': đã có 56/56 câu


Hỏi chatbot:   0%|          | 0/56 [00:00<?, ?it/s]

Đã có kết quả cho 56/56 câu


In [ ]:
run_file = f"{BASE}/runs/{CFG['run_name']}_outputs.jsonl"
n_done = sum(1 for _ in open(run_file, encoding="utf-8"))
print(f"Đã xong {n_done}/{len(bench)} câu")

In [ ]:
import time, traceback
q = bench.question.iloc[0]
print("run_name:", CFG["run_name"], "| model đang dùng:", getattr(llm, "model_name", "?"))
run_file = f"{BASE}/runs/{CFG['run_name']}_outputs.jsonl"
print("Số câu đã lưu trong file:", sum(1 for _ in open(run_file, encoding="utf-8")) if os.path.exists(run_file) else 0)

t = time.time(); cands = vs.similarity_search(q, k=RETRIEVE_N)
print(f"1) FAISS: {time.time()-t:.1f}s ({len(cands)} đoạn)")

t = time.time(); docs = retrieve_with_rerank(q)
print(f"2) Rerank: {time.time()-t:.1f}s ({len(docs)} đoạn giữ lại)")

t = time.time()
try:
    out = chain.invoke({"context": format_docs(docs), "question": q})
    print(f"3) Groq: {time.time()-t:.1f}s")
    print("Trả lời:", out[:300])
except Exception:
    print(f"3) Groq LỖI sau {time.time()-t:.1f}s")
    traceback.print_exc()

## 8. Đánh giá Retrieval
Đo trên các câu có đáp án: chunk đúng (gold) có nằm trong top-k không.
Vì chunking của chatbot có thể khác bộ benchmark, ta so khớp theo **độ trùng lặp văn bản** thay vì so ID.

In [ ]:
def shingles(t, n=6):
    w = t.lower().split()
    return {" ".join(w[i:i+n]) for i in range(max(1, len(w) - n + 1))}

def is_match(ctx, gold, thr=0.4):
    g = shingles(gold)
    return len(g & shingles(ctx)) / max(1, len(g)) >= thr

def retrieval_row(r):
    if not r["answerable"] or not isinstance(r["contexts"], list):
        return pd.Series(dict(hit=np.nan, rr=np.nan))
    for rank, c in enumerate(r["contexts"], 1):
        if is_match(c, r["gold_text"]):
            return pd.Series(dict(hit=1.0, rr=1.0 / rank))
    return pd.Series(dict(hit=0.0, rr=0.0))

df[["hit", "rr"]] = df.apply(retrieval_row, axis=1)
print(f"Hit@{CFG['top_k']}: {df.hit.mean():.3f} | MRR: {df.rr.mean():.3f}")

Hit@5: 0.512 | MRR: 0.428


## 9. Đánh giá Generation
- **RAGAS** (judge Gemini): faithfulness, answer relevancy, context precision, context recall, cho các câu có đáp án.
- **Abstention** (judge Gemini): chatbot có từ chối đúng lúc không (câu không có đáp án) và có từ chối nhầm không (câu có đáp án).

In [ ]:
from ragas import evaluate, EvaluationDataset
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithReference, LLMContextRecall
from ragas.run_config import RunConfig
from langchain_groq import ChatGroq

JUDGE_MODEL = "openai/gpt-oss-120b"     # giám khảo; khác họ với chatbot (qwen)
judge_tag = JUDGE_MODEL.replace("/", "_")

ragas_llm = LangchainLLMWrapper(ChatGroq(model=JUDGE_MODEL, temperature=0, timeout=120, max_retries=2))
ragas_emb = LangchainEmbeddingsWrapper(emb)   # embedding bge-m3 chạy cục bộ, không tốn quota
metrics = [Faithfulness(), ResponseRelevancy(strictness=1),
           LLMContextPrecisionWithReference(), LLMContextRecall()]

ans_df = df[df.answerable & df.answer.notna()].reset_index(drop=True)
# ans_df = ans_df.head(10)    # bỏ dấu # để chạy thử 10 câu trước khi chạy hết

ragas_cache = f"{BASE}/runs/{CFG['run_name']}_ragas_{judge_tag}.csv"
prev = pd.read_csv(ragas_cache) if os.path.exists(ragas_cache) else pd.DataFrame()
parts = [prev] if len(prev) else []
todo = ans_df[~ans_df.id.isin(set(prev["id"]) if len(prev) else set())].reset_index(drop=True)
print(f"Giám khảo: {JUDGE_MODEL} | cần chấm {len(todo)} câu (cache đã có {len(prev)})")

BATCH = 5   # lô nhỏ + lưu cache sau mỗi lô, phòng khi dính giới hạn của Groq
for s0 in range(0, len(todo), BATCH):
    sub = todo.iloc[s0:s0 + BATCH]
    ds = EvaluationDataset.from_list([
        dict(user_input=r.question, response=r.answer, retrieved_contexts=r.contexts, reference=r.ground_truth)
        for r in sub.itertuples()])
    res = evaluate(ds, metrics=metrics, llm=ragas_llm, embeddings=ragas_emb,
                   run_config=RunConfig(max_workers=2, timeout=180, max_retries=10))
    rg = res.to_pandas()
    mc = [c for c in rg.columns if c not in ("user_input", "response", "retrieved_contexts", "reference")]
    parts.append(pd.concat([sub[["id"]].reset_index(drop=True), rg[mc].reset_index(drop=True)], axis=1))
    pd.concat(parts, ignore_index=True).to_csv(ragas_cache, index=False)
    print(f"Xong {min(s0 + BATCH, len(todo))}/{len(todo)}")

scores_df = pd.concat(parts, ignore_index=True)
metric_cols = [c for c in scores_df.columns if c != "id"]
ans_df = ans_df.merge(scores_df, on="id", how="left")
print("Số ô NaN (cần chạy lại nếu nhiều):", int(scores_df[metric_cols].isna().sum().sum()))
scores_df[metric_cols].mean()

/tmp/ipykernel_1491/2418173783.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithReference, LLMContextRecall
/tmp/ipykernel_1491/2418173783.py:4: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithReference, LLMContextRecall
/tmp/ipykernel_1491/2418173783.py:4: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Conte

Giám khảo: openai/gpt-oss-120b | cần chấm 16 câu (cache đã có 25)


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[3]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[8]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[12]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m7qvwndaed9sh4bppeq77af1` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198209, Requested 5270. Please try again in 25m2.928s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
ERROR:ragas.executor:Exception raised in Job[14]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m7qvwndaed9sh4bppeq77af1` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198178, Requested 3652. Please try again in 13m10.56s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.co

Xong 5/16


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

... (5 outputs left out)

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[2]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[3]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m7qvwndaed9sh4bppeq77af1` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199578, Requested 2930. Please try again in 18m3.456s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})


Xong 16/16
Số ô NaN (cần chạy lại nếu nhiều): 93


,0
faithfulness,0.873855
answer_relevancy,0.833776
llm_context_precision_with_reference,0.797917
context_recall,0.933333


In [ ]:
import re, json, time
from groq import Groq
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"], timeout=60.0, max_retries=0)

def _parse(txt):
    txt = re.sub(r"<think>.*?</think>", "", txt, flags=re.S).strip()
    return json.loads(txt[txt.find("{"): txt.rfind("}") + 1])

def chat_json(prompt, temp=0.0):
    for a in range(4):
        try:
            r = groq_client.chat.completions.create(
                model=JUDGE_MODEL, temperature=temp, response_format={"type": "json_object"},
                messages=[{"role": "user", "content": prompt}])
            time.sleep(1.5)
            return _parse(r.choices[0].message.content)
        except Exception as e:
            msg = str(e)
            if "404" in msg or "model_not_found" in msg:
                raise RuntimeError("Sai tên model giám khảo: " + msg[:200])
            print(f"[Groq] lỗi {a+1}/4: {msg[:80]}")
            time.sleep(10 * (a + 1))
    return None

print(chat_json('Trả về JSON: {"ok": true}'))

{'ok': True}


In [ ]:
import numpy as np, pandas as pd
from tqdm.auto import tqdm

def judge_abstain(question, answer):
    p = ("Xác định câu trả lời của chatbot có phải là TỪ CHỐI/NÓI KHÔNG BIẾT/KHÔNG TÌM THẤY thông tin hay không.\n"
         "Nếu chatbot đưa ra thông tin cụ thể để trả lời thì abstained=false.\n"
         'Trả về JSON: {"abstained": true|false}\n\n'
         "CÂU HỎI: " + str(question) + "\nCÂU TRẢ LỜI: " + str(answer))
    out = chat_json(p, temp=0)
    if not out or "abstained" not in out:
        return np.nan                      # giám khảo lỗi: bỏ trống, không tính nhầm
    v = out["abstained"]
    return float(v if isinstance(v, bool) else str(v).strip().lower() == "true")

abs_cache = f"{BASE}/runs/{CFG['run_name']}_abstain_{judge_tag}.csv"
cache = {}
if os.path.exists(abs_cache):
    old = pd.read_csv(abs_cache)
    cache = dict(zip(old["id"].astype(str), old["abstained"]))
print(f"Chấm từ chối bằng {JUDGE_MODEL} | cache đã có {len(cache)} câu")

todo = [r for r in df.itertuples() if isinstance(r.answer, str) and str(r.id) not in cache]
for i, r in enumerate(tqdm(todo, desc="Chấm từ chối")):
    cache[str(r.id)] = judge_abstain(r.question, r.answer)
    if (i + 1) % 20 == 0:
        pd.DataFrame({"id": list(cache), "abstained": list(cache.values())}).to_csv(abs_cache, index=False)
pd.DataFrame({"id": list(cache), "abstained": list(cache.values())}).to_csv(abs_cache, index=False)

df["abstained"] = df["id"].astype(str).map(cache).astype(float)
una = df[~df.answerable]
ans = df[df.answerable]
print("Số câu giám khảo lỗi (bỏ trống):", int(df["abstained"].isna().sum()))
print(f"Từ chối ĐÚNG (câu không có đáp án): {una.abstained.mean():.3f}   <- càng cao càng tốt")
print(f"Từ chối NHẦM (câu có đáp án):       {ans.abstained.mean():.3f}   <- càng thấp càng tốt")

Chấm từ chối bằng openai/gpt-oss-120b | cache đã có 0 câu


Chấm từ chối:   0%|          | 0/56 [00:00<?, ?it/s]

[Groq] lỗi 1/4: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gp
[Groq] lỗi 2/4: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gp
[Groq] lỗi 3/4: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gp
[Groq] lỗi 4/4: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gp
[Groq] lỗi 1/4: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gp
[Groq] lỗi 2/4: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gp
[Groq] lỗi 3/4: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gp
[Groq] lỗi 4/4: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gp
[Groq] lỗi 1/4: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gp
[Groq] lỗi 2/4: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gp
[Groq] lỗi 3/4: Error code: 42

## 10. Tổng hợp, phân tích lỗi và lưu kết quả

In [ ]:
lat = df.latency.dropna()
summary = {
    "run_name": CFG["run_name"], "chatbot_model": CFG["chatbot_model"], "judge_model": CFG["judge_model"],
    "top_k": CFG["top_k"], "chunk_size": CFG["chunk_size"], "n_questions": len(df),
    f"hit@{CFG['top_k']}": df.hit.mean(), "mrr": df.rr.mean(),
    **{m: ans_df[m].mean() for m in metric_cols},
    "abstain_correct": una.abstained.mean() if len(una) else np.nan,
    "false_refusal": ans.abstained.mean() if len(ans) else np.nan,
    "latency_p50_s": lat.median(), "latency_p95_s": lat.quantile(0.95),
}
summary = {k: (round(v, 4) if isinstance(v, float) else v) for k, v in summary.items()}
display(pd.Series(summary))

# Gán nhãn lỗi cho từng câu có đáp án
f_col = [c for c in metric_cols if "faith" in c.lower()][0]
a2 = ans_df.merge(df[["id", "hit", "abstained"]], on="id", how="left", suffixes=("", "_d"))
def label(r):
    if r["abstained"] is True or r["abstained"] == 1: return "từ chối nhầm"
    if r["hit"] == 0: return "retrieval trượt"
    if pd.notna(r[f_col]) and r[f_col] < 0.7: return "bịa/không bám context"
    return "ổn"
a2["loi"] = a2.apply(label, axis=1)
print(a2["loi"].value_counts())
print("\nTheo loại câu hỏi:")
display(a2.groupby("qtype")[[f_col, "hit"]].mean().round(3))

# Lưu
out_xlsx = f"{BASE}/runs/{CFG['run_name']}_result.xlsx"
with pd.ExcelWriter(out_xlsx) as w:
    pd.DataFrame([summary]).to_excel(w, "summary", index=False)
    df.drop(columns=["contexts"]).to_excel(w, "all", index=False)
    a2.drop(columns=["contexts", "retrieved_contexts"], errors="ignore").to_excel(w, "answerable_scores", index=False)
hist = f"{BASE}/history.csv"
pd.DataFrame([summary]).to_csv(hist, mode="a", header=not os.path.exists(hist), index=False)
print("Đã lưu:", out_xlsx, "và", hist)

,0
run_name,bge_m3_hybrid_rerank
chatbot_model,openai/gpt-oss-120b
judge_model,models/gemini-3.8-flash
top_k,5
chunk_size,1500
n_questions,56
hit@5,0.5122
mrr,0.4276
faithfulness,0.8739
answer_relevancy,0.8338


loi
ổn                       20
retrieval trượt          19
bịa/không bám context     1
từ chối nhầm              1
Name: count, dtype: int64

Theo loại câu hỏi:


,faithfulness,hit
qtype,,
"câu hỏi ngắn kiểu người dùng thật: viết tắt, thiếu dấu hoặc thiếu chủ ngữ",NaN,0.250
giải thích khái niệm hoặc quy trình,1.000,0.833
liệt kê các mục/bước/điều kiện,0.881,0.429
tra cứu sự kiện/con số cụ thể,0.881,0.500
tình huống áp dụng điều kiện (nếu... thì...),0.773,0.625


/tmp/ipykernel_1491/3007368662.py:30: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  pd.DataFrame([summary]).to_excel(w, "summary", index=False)
/tmp/ipykernel_1491/3007368662.py:31: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.drop(columns=["contexts"]).to_excel(w, "all", index=False)
/tmp/ipykernel_1491/3007368662.py:32: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  a2.drop(columns=["contexts", "retrieved_contexts"], errors="ignore").to_excel(w, "answerable_scores", index=False)


Đã lưu: /content/drive/MyDrive/ChatboxRAG/rag_benchmark/runs/bge_m3_hybrid_rerank_result.xlsx và /content/drive/MyDrive/ChatboxRAG/rag_benchmark/history.csv


## 11. Vòng lặp tinh chỉnh
1. Giữ nguyên bộ câu hỏi và judge. Đổi **một** thứ ở chatbot (chunk size, top-k, prompt, model Groq, reranker...).
2. Đổi `CFG["run_name"]`, chạy lại từ **mục 6** (mục 3 nếu đổi chunking).
3. So sánh các dòng trong `history.csv`.

| Điểm thấp ở | Nên thử |
|---|---|
| Hit@k / context recall | tăng top-k, chunk size khác, hybrid BM25+vector, reranker, query rewriting |
| Context precision | reranker, giảm top-k |
| Faithfulness | siết prompt hệ thống, model Groq lớn hơn |
| Từ chối nhầm / không từ chối | chỉnh phần hướng dẫn "khi thiếu thông tin" trong prompt |

**Mở rộng sau:** thêm câu multi-hop, hội thoại nhiều lượt, câu từ log thực tế, và tập "test set giữ kín" để tránh overfit.